# 简介

我们最近发布了全新的开源 **Agents SDK**，它旨在通过一个轻量、易用且抽象极少的软件包，帮助你构建智能体式 AI 应用。

本 cookbook 演示如何将 Agents SDK 与 Stripe 的 API 结合使用，来处理争议管理——这是许多企业都会面临的常见运营挑战。具体来说，我们关注两个真实场景：

1. **公司失误：**  
   公司明显犯了错误的场景，例如未能履行订单，此时接受争议是恰当的做法。

2. **客户争议（最终销售）：**  
   客户在收到正确商品、且明知该购买属于最终销售（不退不换）的情况下，仍故意对交易提出争议的场景，需要进一步调查以收集支持性证据。

为了应对这些场景，我们将引入三个不同的智能体：

- **分诊智能体（Triage Agent）：**  
  根据订单的履约状态，决定是接受争议还是将其升级处理。

- **受理智能体（Acceptance Agent）：**  
  通过自动接受争议并给出简明理由，处理情况明确的案例。

- **调查智能体（Investigator Agent）：**  
通过分析沟通记录和订单信息来收集关键证据，对争议进行彻底调查。

在本 cookbook 中，我们将一步步带你了解自定义智能体工作流如何实现争议管理自动化并支持你的业务运营。


## 前提条件

在运行本 cookbook 之前，你必须注册以下账号并完成一些设置操作。这些前提条件对于与本项目使用的 API 进行交互至关重要。

#### 1. OpenAI 账号

- **用途：**  
  你需要一个 OpenAI 账号来访问语言模型，并使用本 cookbook 中介绍的 Agents SDK。

- **操作：**  
  如果你还没有账号，请先[注册 OpenAI 账号](https://openai.com)。拥有账号后，访问 [OpenAI API Keys 页面](https://platform.openai.com/api-keys)创建一个 API 密钥。

#### 2. Stripe 账号

- **用途：**  
  在我们的演示工作流中，需要一个 Stripe 账号来模拟支付处理、管理争议并与 Stripe API 交互。

- **操作：**  
  访问 [Stripe 注册页面](https://dashboard.stripe.com/register)创建一个免费的 Stripe 账号。

- **找到你的 API 密钥：**  
  登录 Stripe 控制台，然后进入 **Developers > API keys**。

- **使用测试模式：**  
  所有开发和测试都请使用你的 **Test Secret Key**。


#### 3. 创建包含 OpenAI API 和 Stripe API 密钥的 .env 文件

```
OPENAI_API_KEY=
STRIPE_SECRET_KEY=
```

### 环境设置
首先我们安装必要的依赖，然后导入库并编写一些稍后会用到的工具函数。

In [ ]:
%pip install python-dotenv --quiet
%pip install openai-agents --quiet
%pip install stripe --quiet
%pip install typing_extensions --quiet

In [211]:
import os
import logging
import json
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool  # Only import what you need
import stripe
from typing_extensions import TypedDict, Any
# Load environment variables from .env file
load_dotenv()

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set Stripe API key from environment variables
stripe.api_key = os.getenv("STRIPE_SECRET_KEY")

#### 定义函数工具
本节定义了几个辅助函数工具，用于支持争议处理工作流。
<br>
 
- `get_order`、`get_phone_logs` 和 `get_emails` 根据提供的标识符返回订单详情以及邮件/电话记录，以此模拟外部数据查询。
- `retrieve_payment_intent` 与 Stripe API 交互，获取支付意图（payment intent）详情。
- `close_dispute` 使用提供的争议 ID 自动关闭 Stripe 争议，确保争议得到妥善解决并被记录。


In [212]:
@function_tool
def get_phone_logs(phone_number: str) -> list:
    """
    Return a list of phone call records for the given phone number.
    Each record might include call timestamps, durations, notes, 
    and an associated order_id if applicable.
    """
    phone_logs = [
        {
            "phone_number": "+15551234567",
            "timestamp": "2023-03-14 15:24:00",
            "duration_minutes": 5,
            "notes": "Asked about status of order #1121",
            "order_id": 1121
        },
        {
            "phone_number": "+15551234567",
            "timestamp": "2023-02-28 10:10:00",
            "duration_minutes": 7,
            "notes": "Requested refund for order #1121, I told him we were unable to refund the order because it was final sale",
            "order_id": 1121
        },
        {
            "phone_number": "+15559876543",
            "timestamp": "2023-01-05 09:00:00",
            "duration_minutes": 2,
            "notes": "General inquiry; no specific order mentioned",
            "order_id": None
        },
    ]
    return [
        log for log in phone_logs if log["phone_number"] == phone_number
    ]


@function_tool
def get_order(order_id: int) -> str:
    """
    Retrieve an order by ID from a predefined list of orders.
    Returns the corresponding order object or 'No order found'.
    """
    orders = [
        {
            "order_id": 1234,
            "fulfillment_details": "not_shipped"
        },
        {
            "order_id": 9101,
            "fulfillment_details": "shipped",
            "tracking_info": {
                "carrier": "FedEx",
                "tracking_number": "123456789012"
            },
            "delivery_status": "out for delivery"
        },
        {
            "order_id": 1121,
            "fulfillment_details": "delivered",
            "customer_id": "cus_PZ1234567890",
            "customer_phone": "+15551234567",
            "order_date": "2023-01-01",
            "customer_email": "customer1@example.com",
            "tracking_info": {
                "carrier": "UPS",
                "tracking_number": "1Z999AA10123456784",
                "delivery_status": "delivered"
            },
            "shipping_address": {
                "zip": "10001"
            },
            "tos_acceptance": {
                "date": "2023-01-01",
                "ip": "192.168.1.1"
            }
        }
    ]
    for order in orders:
        if order["order_id"] == order_id:
            return order
    return "No order found"


@function_tool
def get_emails(email: str) -> list:
    """
    Return a list of email records for the given email address.
    """
    emails = [
        {
            "email": "customer1@example.com",
            "subject": "Order #1121",
            "body": "Hey, I know you don't accept refunds but the sneakers don't fit and I'd like a refund"
        },
        {
            "email": "customer2@example.com",
            "subject": "Inquiry about product availability",
            "body": "Hello, I wanted to check if the new model of the smartphone is available in stock."
        },
        {
            "email": "customer3@example.com",
            "subject": "Feedback on recent purchase",
            "body": "Hi, I recently purchased a laptop from your store and I am very satisfied with the product. Keep up the good work!"
        }
    ]
    return [email_data for email_data in emails if email_data["email"] == email]


@function_tool
async def retrieve_payment_intent(payment_intent_id: str) -> dict:
    """
    Retrieve a Stripe payment intent by ID.
    Returns the payment intent object on success or an empty dictionary on failure.
    """
    try:
        return stripe.PaymentIntent.retrieve(payment_intent_id)
    except stripe.error.StripeError as e:
        logger.error(f"Stripe error occurred while retrieving payment intent: {e}")
        return {}

@function_tool
async def close_dispute(dispute_id: str) -> dict:
    """
    Close a Stripe dispute by ID. 
    Returns the dispute object on success or an empty dictionary on failure.
    """
    try:
        return stripe.Dispute.close(dispute_id)
    except stripe.error.StripeError as e:
        logger.error(f"Stripe error occurred while closing dispute: {e}")
        return {}


### 定义智能体

- **争议受理调查智能体（investigator_agent）** 负责调查争议，收集所有相关证据并提供报告。
- **接受争议智能体（accept_dispute_agent）** 处理被判定为合理的争议，自动关闭它们并对决定做简要说明。
- **分诊智能体（triage_agent）** 担任决策者：从支付意图的元数据中提取订单 ID，检索详细的订单信息，然后决定是将争议升级给调查智能体，还是交给接受争议智能体。
- 这些智能体共同构成一个模块化工作流，通过将特定任务委派给专门的智能体，实现争议解决流程的自动化和精简。


In [213]:
investigator_agent = Agent(
    name="Dispute Intake Agent",
    instructions=(
        "As a dispute investigator, please compile the following details in your final output:\n\n"
        "Dispute Details:\n"
        "- Dispute ID\n"
        "- Amount\n"
        "- Reason for Dispute\n"
        "- Card Brand\n\n"
        "Payment & Order Details:\n"
        "- Fulfillment status of the order\n"
        "- Shipping carrier and tracking number\n"
        "- Confirmation of TOS acceptance\n\n"
        "Email and Phone Records:\n"
        "- Any relevant email threads (include the full body text)\n"
        "- Any relevant phone logs\n"
    ),
    model="o3-mini",
    tools=[get_emails, get_phone_logs]
)


accept_dispute_agent = Agent(
    name="Accept Dispute Agent",
    instructions=(
        "You are an agent responsible for accepting disputes. Please do the following:\n"
        "1. Use the provided dispute ID to close the dispute.\n"
        "2. Provide a short explanation of why the dispute is being accepted.\n"
        "3. Reference any relevant order details (e.g., unfulfilled order, etc.) retrieved from the database.\n\n"
        "Then, produce your final output in this exact format:\n\n"
        "Dispute Details:\n"
        "- Dispute ID\n"
        "- Amount\n"
        "- Reason for Dispute\n\n"
        "Order Details:\n"
        "- Fulfillment status of the order\n\n"
        "Reasoning for closing the dispute\n"
    ),
    model="gpt-4o",
    tools=[close_dispute]
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "Please do the following:\n"
        "1. Find the order ID from the payment intent's metadata.\n"
        "2. Retrieve detailed information about the order (e.g., shipping status).\n"
        "3. If the order has shipped, escalate this dispute to the investigator agent.\n"
        "4. If the order has not shipped, accept the dispute.\n"
    ),
    model="gpt-4o",
    tools=[retrieve_payment_intent, get_order],
    handoffs=[accept_dispute_agent, investigator_agent],
)


#### 检索争议并启动智能体工作流
该函数使用提供的 `payment_intent_id` 从 Stripe 检索争议详情，并将检索到的争议信息传给指定的 `triage_agent`，从而启动争议处理工作流。


In [214]:
async def process_dispute(payment_intent_id, triage_agent):
    """Retrieve and process dispute data for a given PaymentIntent."""
    disputes_list = stripe.Dispute.list(payment_intent=payment_intent_id)
    if not disputes_list.data:
        logger.warning("No dispute data found for PaymentIntent: %s", payment_intent_id)
        return None
    
    dispute_data = disputes_list.data[0]
    
    relevant_data = {
        "dispute_id": dispute_data.get("id"),
        "amount": dispute_data.get("amount"),
        "due_by": dispute_data.get("evidence_details", {}).get("due_by"),
        "payment_intent": dispute_data.get("payment_intent"),
        "reason": dispute_data.get("reason"),
        "status": dispute_data.get("status"),
        "card_brand": dispute_data.get("payment_method_details", {}).get("card", {}).get("brand")
    }
    
    event_str = json.dumps(relevant_data)
    # Pass the dispute data to the triage agent
    result = await Runner.run(triage_agent, input=event_str)
    logger.info("WORKFLOW RESULT: %s", result.final_output)
    
    return relevant_data, result.final_output

#### 场景 1：公司失误（未收到商品）
该场景代表公司明显犯了错误的情况——例如未能履行订单或未发货。在这种情况下，接受争议可能比提出异议更合适。

In [ ]:
payment = stripe.PaymentIntent.create(
  amount=2000,
  currency="usd",
  payment_method = "pm_card_createDisputeProductNotReceived",
  confirm=True,
  metadata={"order_id": "1234"},
  off_session=True,
  automatic_payment_methods={"enabled": True},
)
relevant_data, triage_result = await process_dispute(payment.id, triage_agent)

#### 场景 2：客户争议（最终销售）
该场景描述的是客户在收到正确商品、并且完全清楚该购买已明确标注为“最终销售”（不退款、不退货）的情况下，仍故意对交易提出争议。此类争议通常需要进一步调查以收集证据，才能有效地对争议提出异议。

In [ ]:
payment = stripe.PaymentIntent.create(
  amount=2000,
  currency="usd",
  payment_method = "pm_card_createDispute",
  confirm=True,
  metadata={"order_id": "1121"},
  off_session=True,
  automatic_payment_methods={"enabled": True},
)
relevant_data, triage_result = await process_dispute(payment.id, triage_agent)

## 结论

在这个 Jupyter Notebook 中，我们探索了 **OpenAI Agents SDK** 的能力，演示了如何采用简单、以 Python 为先的方式高效创建基于智能体的 AI 应用。具体而言，我们展示了以下 SDK 功能：

- **智能体循环（Agent Loop）**：管理工具调用，将结果传回 LLM，并循环直至完成。
- **交接（Handoffs）**：支持多个专门智能体之间的协调与任务委派。
- **函数工具（Function Tools）**：将 Python 函数转换为工具，并自动生成和校验 schema。

此外，SDK 还内置了 **追踪（Tracing）** 功能，可通过 OpenAI 控制台访问。追踪可以帮助你在开发和生产阶段可视化、调试和监控智能体工作流。它还能与 OpenAI 的评估、微调和蒸馏工具无缝集成。

虽然本 notebook 没有直接介绍，但强烈建议在生产应用中实现 **护栏（Guardrails）**，以校验输入并主动发现错误。

总的来说，本 notebook 为进一步探索打下了清晰的基础，突出展示了 OpenAI Agents SDK 如何让智能体驱动的工作流变得直观而高效。